# NLP Robot Command Parser

## 0. Setup
Prepares the environment. The repository is cloned from GitHub, and the SCAN dataset is downloaded. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [ ]:
# Clone repo and move into it 
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

In [ ]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {cfg["model"]["name"]}')

## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration. 

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'

train_data, test_data = load_scan(
    split     = cfg['data']['scan_split'],
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language : {language}')
print(f'Train examples : {len(train_data)}')
print(f'Test  examples : {len(test_data)}')
print(f'First example  : {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer = get_tokenizer(cfg['model']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test' : to_hf_dataset(test_data),
})

tokenized_dataset = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset)